In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel
from typing import TypedDict
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
llm = ChatOpenAI()

In [4]:
def chat_node(state:MessagesState):
    """Invokes LLM to get a response based on user input"""

    messages = state['messages']
    response = llm.invoke(messages)

    return {'messages': [response]}

import time
def search_node(state:MessagesState):
    """Dummy search node to send server events for searching"""
    time.sleep(5)
    return {}

def call_tools(state:MessagesState):
    """Dummy tool node to send server events for tool calls"""
    time.sleep(5)
    return {}

In [5]:
builder = StateGraph(MessagesState)

config = {"configurable": {"thread_id": "thread-1"}}

builder.add_node('chat_node', chat_node)
builder.add_node('search_node', search_node)
builder.add_node('call_tools', call_tools)

builder.add_edge(START, 'search_node')
builder.add_edge('search_node', 'call_tools')
builder.add_edge('call_tools', 'chat_node')
builder.add_edge('chat_node', END)

checkpointer = InMemorySaver()

chatbot = builder.compile(checkpointer=checkpointer)

In [6]:
response = chatbot.invoke({'messages': HumanMessage(content="Generate a 3 line poem on AI")}, config=config)
print(response['messages'][-1].content)

The AI's mind, a vast expanse of code
Unraveling mysteries, on a digital road
A marvel of technology, forever to unfold
